In [ ]:
import random
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_variables import *
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_types import carry_out_comparative_experiment_without_ground_truth_and_multiple_choices
from vpei.epistemic_consistency.experiment_utils import print_comparative_experiment_results, build_user_prompt_template_with_variable_repeats

cvs_df = pd.read_csv('./data/cvs.csv')
jd_df = pd.read_csv('./data/job_descriptions.csv')

# One representative job description per profession (first row per profession)
shared_jd = jd_df.groupby('profession')['job_description'].first().to_dict()

# Merged DataFrame: each CV row paired with its profession's shared job description
df = pd.DataFrame({
    'profession': cvs_df['profession'],
    'cv': cvs_df['cv'],
    'job_description': cvs_df['profession'].map(shared_jd),
})
df

In [ ]:
experiment_name = "cvs"
number_of_choices = 5
system_prompt = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth_and_multiple_choices"]["system_prompt"]
user_prompt_template_repeated_block = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth_and_multiple_choices"]["user_prompt_template_repeated_block"]
user_prompt_template_repeated_attribution_block = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth_and_multiple_choices"].get("user_prompt_template_repeated_attribution_block", None)
user_prompt_template_prefix = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth_and_multiple_choices"].get("user_prompt_template_prefix", None)
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template_prefix)
print("---")
print(build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block))

In [ ]:
models = ["gpt-5-mini"]

n = 10
number_of_choices = 5
stimuli_factors = ["cv", "job_description"]
additional_variables_from_df_to_save = ["profession"]
custom_model_kwargs = {}
path_to_save_model_outputs = "./comparative_experiment_without_ground_truth_and_multiple_choices"
random_seed = 42

In [ ]:
# Sampler: always draws number_of_choices CVs from the same randomly chosen profession
def within_profession_sampler(seed):
    profession = random.choice(df['profession'].unique().tolist())
    return df[df['profession'] == profession].sample(number_of_choices, random_state=seed)

In [ ]:
payloads = await carry_out_comparative_experiment_without_ground_truth_and_multiple_choices(
    models=models,
    df=df,
    n=n,
    system_prompt=system_prompt,
    user_prompt_template_repeated_block=user_prompt_template_repeated_block,
    user_prompt_template_repeated_attribution_block=user_prompt_template_repeated_attribution_block,
    user_prompt_template_prefix=user_prompt_template_prefix,
    stimuli_factors=stimuli_factors,
    additional_variables_from_df_to_save=additional_variables_from_df_to_save,
    custom_model_kwargs=custom_model_kwargs,
    path_to_save_model_outputs=path_to_save_model_outputs,
    number_of_choices=number_of_choices,
    random_seed=random_seed,
    df_sampler=within_profession_sampler,
)

print_comparative_experiment_results(payloads, models)